# 数据准备与核查

读取已保存的 Chicago Food Inspections 快照，离线生成历史表、带标签表和质量报告。
本 Notebook 只完成数据准备；课程要求的完整 EDA、建模计划与模型训练仍需后续完成。

本次检查的 Results / Violations 只能用于分析或目标定义，不能作为预测本次检查的输入。

In [1]:
from pathlib import Path
import json
import os
import subprocess
import sys

ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "config/data.json").is_file() and (p / "scripts/prepare_data.py").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Please start this notebook from the project directory.")
os.environ.setdefault("MPLCONFIGDIR", str(ROOT / "tmp/matplotlib"))
import pandas as pd
from IPython.display import display

config = json.loads((ROOT / "config/data.json").read_text(encoding="utf-8"))
pd.set_option("display.max_columns", 30)


## 1. 从保存的原始数据重新准备

本步骤使用标准库执行本地脚本，不联网、不安装依赖、不训练模型。它验证原始文件校验和、下载时间和允许的数据范围，随后生成派生表并校验输出。

In [2]:
completed = subprocess.run(
    [sys.executable, "-X", "utf8", str(ROOT / "scripts/prepare_data.py")],
    cwd=ROOT, check=True, capture_output=True, text=True, encoding="utf-8",
)
print(completed.stdout)
quality = json.loads((ROOT / "reports/data_quality.json").read_text(encoding="utf-8"))


Prepared 50,000 raw rows
Prepared 100,000 raw rows
Prepared 150,000 raw rows
Saved 197,882 history rows; 167,538 labelled rows; 37,863 Fail
Verified all input/output SHA-256 checksums and 167,538 chronologically ordered binary labels.



## 2. 数据来源、实际范围和标签

请求的截止日期并不意味着那一天的数据已经完整上报。这里展示快照实际日期，以及有标签和无标签记录数量。

In [3]:
source = quality["source"]
display(pd.DataFrame([
    ("source", source["portal_page"]),
    ("downloaded_at_utc", source["downloaded_at_utc"]),
    ("requested_start", source["requested_window"]["start_date"]),
    ("requested_end", source["requested_window"]["end_date"]),
    ("actual_start", quality["date_range"]["start"]),
    ("actual_end", quality["date_range"]["end"]),
    ("raw_rows", quality["raw_rows"]),
    ("history_rows", quality["history_rows"]),
    ("labelled_rows", quality["labelled_rows"]),
    ("unlabelled_history_rows", quality["unlabelled_rows"]),
], columns=["item", "value"]))
display(pd.read_csv(ROOT / "reports/label_counts.csv"))


,item,value
0,source,https://data.cityofchicago.org/Health-Human-Se...
1,downloaded_at_utc,2026-10-12T07:59:09+00:00
2,requested_start,2016-01-01
3,requested_end,2026-10-12
4,actual_start,2016-01-04
5,actual_end,2026-10-09
6,raw_rows,197882
7,history_rows,197882
8,labelled_rows,167538
9,unlabelled_history_rows,30344


,label,count,percent
0,0,129675,77.400351
1,1,37863,22.599649


## 3. 安全读取带标签表

编号、许可证和 ZIP 作为字符串读取，以保留格式。日期明确解析为 datetime。
带标签表也含店铺分组键等辅助列；只从字段可用性报告中选择候选特征。

In [4]:
labelled = pd.read_csv(
    ROOT / config["labelled_data"], keep_default_na=False, na_values=[""],
    dtype={
        "inspection_id": "string", "license_number": "string",
        "zip": "string", "establishment_key": "string",
    },
)
labelled["inspection_date"] = pd.to_datetime(labelled["inspection_date"], format="%Y-%m-%d")
labelled["label"] = labelled["label"].astype("int8")
assert len(labelled) == quality["labelled_rows"]
assert set(labelled["label"].unique()) <= {0, 1}
assert labelled["inspection_id"].is_unique
assert labelled["inspection_date"].is_monotonic_increasing
assert not {"results", "violations"} & set(labelled.columns)
feature_columns = quality["default_feature_columns"]
assert set(feature_columns) <= set(labelled.columns)
assert not {"label", "results", "violations"} & set(feature_columns)
print(f"{len(labelled):,} labelled rows; {len(feature_columns)} static/calendar candidate features")
display(labelled.head(5))
display(labelled.dtypes.rename("dtype").to_frame())


167,538 labelled rows; 12 static/calendar candidate features


,inspection_id,dba_name,aka_name,license_number,facility_type,risk,address,city,state,zip,inspection_date,inspection_type,latitude,longitude,risk_level,establishment_key,inspection_year,inspection_month,inspection_weekday,label
0,1609259,YUMMY BUFFET,YUMMY BUFFET,2069749,RESTAURANT,RISK 1 (HIGH),3921-3925 W LAWRENCE AVE,CHICAGO,IL,60625,2016-01-04,COMPLAINT RE-INSPECTION,41.968147,-87.726499,1.0,license:2069749,2016,1,0,0
1,1425220,FABCAKES,FABCAKES,2133991,RESTAURANT,RISK 1 (HIGH),714 N WELLS,CHICAGO,IL,60654,2016-01-05,COMPLAINT,41.895336,-87.634314,1.0,license:2133991,2016,1,1,0
2,1425221,"HOOTERS OF WELLS ST., INC.",HOOTERS,40853,RESTAURANT,RISK 1 (HIGH),658-660 N WELLS ST,CHICAGO,IL,60610,2016-01-05,CANVASS,41.894027,-87.634281,1.0,license:40853,2016,1,1,1
3,1442284,IHOP,IHOP,50037,RESTAURANT,RISK 1 (HIGH),3760 N HALSTED ST,CHICAGO,IL,60613,2016-01-05,CANVASS,41.950901,-87.649772,1.0,license:50037,2016,1,1,0
4,1442285,TACO & BURRITO HOUSE UNO,TACO & BURRITO HOUSE,2374675,RESTAURANT,RISK 1 (HIGH),3939 N BROADWAY,CHICAGO,IL,60613,2016-01-05,CANVASS,41.953922,-87.649956,1.0,license:2374675,2016,1,1,1


,dtype
inspection_id,string
dba_name,str
aka_name,str
license_number,string
facility_type,str
risk,str
address,str
city,str
state,str
zip,string


## 4. 缺失值和字段可用性

缺失类别使用 UNKNOWN，不推测地址信息，也不在全数据上拟合统计填补。
原始历史表包含结果和违规描述；必须遵守当前值与历史值的使用边界。

In [5]:
missing = pd.read_csv(ROOT / "reports/missing_values.csv")
display(missing[missing["table"] == "raw"].sort_values("missing_percent", ascending=False))
availability = pd.read_csv(ROOT / "reports/feature_availability.csv")
display(availability)


,table,column,missing_count,missing_percent
13,raw,Violations,56409,28.506383
4,raw,Facility Type,1030,0.520512
15,raw,Longitude,689,0.348187
16,raw,Location,689,0.348187
14,raw,Latitude,689,0.348187
2,raw,AKA Name,500,0.252676
7,raw,City,125,0.063169
8,raw,State,73,0.036891
5,raw,Risk,39,0.019709
9,raw,Zip,12,0.006064


,column,role,available_before_current_inspection,rule
0,inspection_id,metadata,yes,"Identifier, not a continuous numeric feature."
1,dba_name,metadata,yes,Use only if supplied/known before inspection; ...
2,aka_name,metadata,yes,Use only if supplied/known before inspection; ...
3,license_number,metadata,yes,"Identifier, not a continuous numeric feature."
4,facility_type,candidate_feature,yes,Use only if supplied/known before inspection; ...
5,risk,candidate_feature,yes,Use only if supplied/known before inspection; ...
6,address,metadata,yes,Use only if supplied/known before inspection; ...
7,city,candidate_feature,yes,Use only if supplied/known before inspection; ...
8,state,candidate_feature,yes,Use only if supplied/known before inspection; ...
9,zip,candidate_feature,yes,Use only if supplied/known before inspection; ...


## 5. 重复检查与初步店铺识别

同一家店多次检查是合理历史记录，不能作为重复数据删除。分组优先用有效许可证，回退到名称和地址；这些键仍需核查，尤其是许可证跨地址及同日检查。

In [6]:
checks = {
    name: quality[name] for name in [
        "duplicate_id_rows", "exact_duplicate_rows_removed",
        "establishment_groups", "repeated_establishment_groups",
        "licenses_with_multiple_addresses", "same_day_multiple_inspection_groups",
        "same_day_conflicting_label_groups",
    ]
}
display(pd.Series(checks, name="count").to_frame())
display(pd.DataFrame.from_dict(quality["category_cardinality"], orient="index"))
display(pd.read_csv(ROOT / "reports/annual_counts.csv"))


,count
duplicate_id_rows,0
exact_duplicate_rows_removed,0
establishment_groups,32930
repeated_establishment_groups,25757
licenses_with_multiple_addresses,180
same_day_multiple_inspection_groups,1371
same_day_conflicting_label_groups,57


,raw,normalized
Facility Type,369,337
Risk,5,5
Inspection Type,23,22
Results,7,7
City,70,67
State,8,8


,year,history_rows,labelled_rows,fail_rows
0,2016,22818,19860,4298
1,2017,21587,18499,4121
2,2018,17192,14849,3318
3,2019,19052,16121,3612
4,2020,15123,12600,2935
5,2021,15868,13139,3035
6,2022,16936,14286,3554
7,2023,18261,15413,3835
8,2024,19228,15995,3481
9,2025,19205,16060,3391


## 后续工作

- 阅读 reports/data_preparation.md 中的清理规则和需要核查的情况。
- 完成约 6–8 张有解释的探索图及有证据的数据质量分析。
- 按时间制定训练、验证和最终留出测试划分，并说明店铺重叠情况。
- 历史结果特征仅使用严格早于当前检查日期的记录；同日先后未知。
- 若在 2026-10-30 截止前更新快照，从仓库根目录运行 python scripts/download_data.py --overwrite，再重新运行本 Notebook。
- 隐藏测试输入只使用教学团队提供的文件。
